# 8 · Materials, boundaries & labels

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=08-materials-boundaries.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/08-materials-boundaries.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>


Labels are the *language of "where"* in NGSolve: every **material** (subdomain)
and every **boundary** can carry a name, and we use those names to assign
coefficients, boundary conditions and integration regions. To make it tasty we
bake a 🍪 **cookie with chocolate chips** — two materials in one domain.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import WorkPlane, Axes, Z, Glue, OCCGeometry
from ngsolve import (Mesh, H1, BilinearForm, LinearForm, GridFunction,
                     grad, dx, CF, Integrate)
from ngsolve.webgui import Draw

## 1. A geometry with names

We give the chips the **material** name `"chip"` and the rest `"dough"`, and we
name the outer **boundary** `"rim"`. `Glue` keeps the chips as separate
subdomains inside the cookie.

In [ ]:
def cookie_with_chips():
    R = 3.0
    cookie = WorkPlane().Circle(0, 0, R).Face()
    cookie.edges.name = "rim"                       # name the outer boundary
    chips = []
    for (cx, cy, r) in [(-1.2, 0.8, 0.5), (1.0, 1.1, 0.45), (0.2, -1.3, 0.55),
                        (1.5, -0.8, 0.4), (-1.3, -0.9, 0.45)]:
        chip = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, r).Face()
        chip.name = "chip"                          # material name
        chips.append(chip)
    dough = cookie
    for chip in chips:
        dough = dough - chip
    dough.name = "dough"
    return Glue([dough] + chips)

mesh = Mesh(OCCGeometry(cookie_with_chips(), dim=2).GenerateMesh(maxh=0.4))
print("materials :", set(mesh.GetMaterials()))
print("boundaries:", set(mesh.GetBoundaries()))

## 2. Selecting regions

`mesh.Materials("...")` and `mesh.Boundaries("...")` select named regions
(regular expressions are allowed). We can measure them with `Integrate`.

In [ ]:
print("dough area :", round(Integrate(CF(1), mesh, definedon=mesh.Materials("dough")), 2))
print("chip  area :", round(Integrate(CF(1), mesh, definedon=mesh.Materials("chip")), 2))

## 3. A piecewise coefficient

`mesh.MaterialCF` builds a `CoefficientFunction` that takes a different value
in each material — here the heat conductivity: chocolate conducts much better
than dough.

In [ ]:
kappa = mesh.MaterialCF({"chip": 50.0, "dough": 1.0})
Draw(kappa, mesh)

## 4. Using the labels in a PDE

We heat the cookie uniformly and hold the `rim` at zero. The conductivity
`kappa` enters the bilinear form; `dirichlet="rim"` uses the boundary name.
Notice how the temperature is flatter over the well-conducting chips.

In [ ]:
fes = H1(mesh, order=2, dirichlet="rim")
u, v = fes.TnT()
a = BilinearForm(kappa*grad(u)*grad(v)*dx).Assemble()
f = LinearForm(1*v*dx).Assemble()
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
Draw(gfu)

## 5. A subtle bug — the misspelled region

A wrong region name is **not** an error in NGSolve — it simply selects
*nothing*, and the corresponding integral is silently zero. This is a classic
trap: your code runs, but the result is wrong.

In [ ]:
buggy = mesh.Materials("chips")                     # typo: "chips" instead of "chip"
print("area of Materials('chips'):", Integrate(CF(1), mesh, definedon=buggy))   # -> 0.0 !

:::{dropdown} 🧠 Quiz — why did `Materials("chips")` give 0 and not an error?
Region selectors are matched (as regular expressions) against the **actual**
names; an unmatched pattern yields the *empty* region, whose measure is 0. Always
sanity-check with `mesh.GetMaterials()` / `mesh.GetBoundaries()` (and a quick
`Integrate(CF(1), ...)` on the region) before trusting a result. The fix here is
simply `mesh.Materials("chip")`.
:::

Next we take materials, boundaries and interfaces into **3D** — a ceramic cup
filled with coffee.